# 팀 프로젝트2 — 14조: 다음 학년도 중도탈락 고위험 학과는 어디인가
학업지속 지원 대상 학과 선정을 위한 대학알리미 학과별 중도탈락 패널(2023~2026) 분석 및 예측

- 팀: 정승엽, 박유진, 신효주 · 담당교수 박주원
- Colab 정본 링크: `[팀 입력]`
- 실행 방법: CSV를 올린 뒤 **런타임 → 다시 시작 및 모두 실행**. 처음부터 재실행할 수 있어야 한다(불가하면 −5점).
- 구성: 안내문 5단계 **① 문제 정의 → ② 수집 → ③ 정제 → ④ 분석 → ⑤ 발표**, 그리고 부록(재현성 확인). 기준 문서는 `docs/기준_정리.md`.
- **출처 규칙**: 수치를 출력하는 셀에는 맨 위 주석에 셀 ID(예: `s3-06`)를 둔다. 보고서·발표에 옮길 때 `(출처: 노트북 §③ 셀 s3-06)` 형식으로 붙인다. 출처 없는 수치는 건당 −2점.
- **고정 규칙**: 집단별 탈락률은 `hypotheses-locked-v2` 태그(2026-11-11) 이후에만 계산한다. 분류는 데이터 컬럼만 쓰고 수작업 매핑은 만들지 않는다.

## 0. 환경·경로

In [ ]:
# s0-01 환경·경로
# TODO: import (pandas, numpy, matplotlib, koreanize_matplotlib, sklearn, statsmodels, scipy)
# TODO: 라이브러리 버전 출력 (logs/재현성_확인표.md에 기록)
# TODO: CSV 경로 후보 리스트 (Colab /content/ → 로컬 ../data/raw/) → 존재하는 첫 경로 사용, 없으면 FileNotFoundError
# TODO: RANDOM_STATE = 42
# TODO: 되돌림 스위치 (logs/정제로그.md 표 6과 이름 일치)
#   SCOPE        : 'general_day'(일반 대학 주간, 사전 고정) vs 'all'(전체 표본, 민감도 ②)
#   MIN_ENROLL   : 직전연도 최소 재적 (사전 고정 30, 민감도 10·50)
#   HIGH_PCT     : 고위험 컷 백분위 (사전 고정 75, 민감도 80·70 = 상위 20%·30%)
#   B_SUBFIELDS  : ['기타유럽어・문학', '기타아시아어・문학'] (민감도 ①: + 국제지역학 소계열)
#   RATE_SOURCE  : 공시 비율 vs 계÷재적 재계산 (팀 결정 후)
#   K_SUPPORT    : 지원 가능 학과 수 k [팀 입력]
# TODO: FIG_DIR = 'figures/' 생성, 그림 저장 함수

## ① 문제 정의 (15점)
- **무엇을**: 의사결정 질문, 가설 A·B, 판정 규칙, 반대 결과 처리 규칙을 인용한다(코드 없음).
- **왜**: 결과를 보기 전에 고정했다는 것을 보여 주려고. 이 셀은 수정하지 않는다.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: git 태그 `hypotheses-locked`(v1, 2026-11-08), `hypotheses-locked-v2`(2026-11-11), `00_문제정의_가설.md` 변경 이력

> `00_문제정의_가설.md` v2 인용 (요약). 상세 판정 규칙은 원문을 따른다.

- **누가**: 한국외국어대학교 학사종합지원센터 (보조: 입학처)
- **무엇을**: 다음 학년도 학업지속 지원(상담·학습지원·복학 안내)을 우선 배치할 학과
- **질문 ①**: 전국 일반 대학 학과 중 어떤 설립유형·지역에서 고위험(상위 25%)이 많은가
- **질문 ②**: 그중 한국외대에서는 어떤 학과군이 고위험인가
- **가설 A (전국 구조)**: 사립·비수도권 학과는 고위험 가능성이 높다.
  - 판정: 대계열·전년도 재적을 통제한 로지스틱 회귀에서 두 오즈비가 모두 1보다 크고 95% CI가 1을 포함하지 않으면 맞음. 하나만이면 부분 맞음, 둘 다 아니면 기각.
- **가설 B (한국외대)**: 비주류 외국어 학과군(소계열 기타유럽어・문학·기타아시아어・문학)은 교내 다른 학과보다 탈락률이 높다.
  - 판정: 2024~2026 합산 비율 차이의 부트스트랩 95% 구간 하한이 0보다 크면 맞음. 아니면 기각하고 '모집(입학처)'으로 재정의.
- **예측 절**: A·B 안에서 모델과 기준선 2개를 같은 k의 정밀도로 비교한다. 판정 임계값 없이 정직하게 보고한다.
- **반대 결과**: 그대로 받아들이고 결론에 맞았다/부분 맞았다/틀렸다를 명시한다.

## ② 수집
- **무엇을**: 원본을 읽고 출처·URL·수집일·이용 조건, 행·열 수, 측정 척도를 출력한다.
- **왜**: 데이터가 공개·실제 자료이고 계획서의 규모와 같다는 것을 확인하려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 기대 행·열 수 assert, `docs/수집_기록표.md`와 출력 대조

In [ ]:
# s2-01 출처 표기
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §② 셀 s2-01)'를 붙인다.
# 내용: 공공데이터포털 15164570, 교육부 교육데이터정책과, 공공누리 제1유형(출처 표시), 포털 수정일 2026-09-10, 파일명 날짜 20260831, 다운로드일 [팀 입력]
# TODO

In [ ]:
# s2-02 원본 읽기
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §② 셀 s2-02)'를 붙인다.
# 내용: df_raw = read_csv(encoding='utf-8-sig'). 원본 df는 이후 수정하지 않음
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): shape == (56340, 32)
# TODO

In [ ]:
# s2-03 공시연도별 행 수
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §② 셀 s2-03)'를 붙인다.
# 내용: 공시연도별 행 수
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 2023 14,350 / 2024 13,628 / 2025 13,928 / 2026 14,434
# TODO

In [ ]:
# s2-04 컬럼 개요와 측정 척도
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §② 셀 s2-04)'를 붙인다.
# 내용: 컬럼별 dtype, 고유값 수, 예시값, 측정 척도(명목/등간/비율) 표
# TODO

## ③ 정제 (`logs/정제로그.md` 표 2 순서)
- **무엇을**: 품질 6대 기준으로 사전 진단하고, 범위를 확정하고, 단계별로 이슈를 표시(flag)·추적한다.
- **왜**: 어떤 이슈가 분석 범위·표본에 영향을 주는지 먼저 보고 처리 여부를 정하려고. 행은 지우지 않고 표시 후 표본에서 제외한다.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 기대값 assert, 단계별 추적표(정제로그 표 5), 처리·건수·근거·영향 로그(표 4), 0행 필터 검사

In [ ]:
# s3-00 단계별 추적 도구
# [기준] 행 삭제 금지, y(비율) 보정 금지, 건수는 코드 계산값만, 0행 필터는 멈춤(assert)
# TODO: make_flags(d)  : 이슈별 불리언 표 (재적=0, 재적=0&계>0, 계>재적, 재계산 불일치, 사유합≠계, 비율 결측, 계 결측, 폐교, 신설, N.C.E.)
# TODO: count_table(d) : 이슈 × 단계(원본 / 중복 확인 후 / 범위 안 / 연결 쌍 / 분석 표본) 건수
# TODO: log = []       : 처리·건수·근거·영향 행을 코드로 쌓음

### ③-0 사전 진단 (품질 6대 기준, 읽기 전용)

In [ ]:
# s3-01 [완전성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-01)'를 붙인다.
# 내용: 비율 결측, 계 결측, 연도별 결측률
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 6,372 = 5,569 + 803 / 2023 0.8%, 2024~2026 약 15%
# TODO

In [ ]:
# s3-02 [유일성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-02)'를 붙인다.
# 내용: 완전 중복, 키(공시연도·학교코드·학과코드) 중복
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 0 / 0
# TODO

In [ ]:
# s3-03 [유효성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-03)'를 붙인다.
# 내용: 재적=0, 재적=0이면서 계>0, 계>재적
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 2,500 / 1,497 / 172
# TODO

In [ ]:
# s3-04 [정확성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-04)'를 붙인다.
# 내용: 공시 비율 vs 계÷재적 재계산 불일치
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 68
# TODO

In [ ]:
# s3-05 [일관성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-05)'를 붙인다.
# 내용: 사유합≠계, 학과명 공백, 표준분류 일관성(같은 학과명 5개교 이상 508개 중 중계열 갈림 135·소계열 갈림 232, N.C.E. 0.4/4.8/9.4%, 한국외대 N.C.E. 35), 한국외대 코드 변동
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 0 / 1 / 표준분류 수치 / 신규 38·소멸 24
# TODO

In [ ]:
# s3-06 [적시성]
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-06)'를 붙인다.
# 내용: 학교상태 폐교·신설, 패널 연결 가능성
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 폐교 493 / 신설 263, 4개년 존재 9,369, 쌍 29,071
# TODO

### ③-1 ~ ③-10 정제 단계 (정제로그 표 2와 같은 번호)

In [ ]:
# s3-11 ③-1 중복
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-11)'를 붙인다.
# 내용: 완전 중복 → 키 중복 (0이면 '처리 없음'으로 로그)
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 0 / 0
# TODO

In [ ]:
# s3-12 ③-2 공백·표기
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-12)'를 붙인다.
# 내용: 컬럼명·문자열 값 strip
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 컬럼 32개, 학과명 공백 1건
# TODO

In [ ]:
# s3-13 ③-3 타입 확인
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-13)'를 붙인다.
# 내용: 숫자 컬럼 변환 시 새 결측 여부
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): NaN 증가 0
# TODO

In [ ]:
# s3-14 ③-4 논리 모순 표시(삭제 금지)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-14)'를 붙인다.
# 내용: 재적=0, 재적=0&계>0, 계>재적, 재계산 불일치, 사유합≠계
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 2,500 / 1,497 / 172 / 68 / 0
# TODO

In [ ]:
# s3-15 ③-5 결측 분해
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-15)'를 붙인다.
# 내용: 비율 결측 = 계 결측 + 재적 0. 보정 안 함
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 6,372 = 5,569 + 803
# TODO

In [ ]:
# s3-16 ③-6 범위 확정 (일반 대학 주간)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-16)'를 붙인다.
# 내용: in_scope = 학교종류=='대학교' & 주야간구분=='주간' & 학과특성=='일반과정' & 학교상태!='폐교'. 행 삭제 없이 표시. 조건별 단독·순차 제외 건수(수집_기록표 표)
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 50,081행, 연도별 12,676 / 12,116 / 12,388 / 12,901, 210개교
# TODO

In [ ]:
# s3-17 ③-7 표준분류 N.C.E. 표시
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-17)'를 붙인다.
# 내용: 대·중·소계열 N.C.E. 건수(범위 안/밖, 한국외대 별도). 재분류 안 함
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 0.4% / 4.8% / 9.4% (전체 기준)
# TODO

In [ ]:
# s3-18 ③-8 패널 연결
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-18)'를 붙인다.
# 내용: 범위 안에서 (학교코드, 학과코드) 전년→당해 쌍. 결합 전후 행 수 기록. 한국외대 미연결 건수
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): (노트북 실행 후 확정)
# TODO

In [ ]:
# s3-19 ③-9 분석 표본 확정
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-19)'를 붙인다.
# 내용: 직전연도 재적≥MIN_ENROLL, 비율 유효. 민감도 표본(10·50, 전체 표본) 함께 생성
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 단계별 추적표
# TODO

In [ ]:
# s3-20 ③-10 손대지 않는 것·최종 점검
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §③ 셀 s3-20)'를 붙인다.
# 내용: 정제로그 표 8 항목 확인, 필터 전후 행 수, 0행 무음 실패 검사, 추적표·로그 출력
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): assert 통과
# TODO

In [ ]:
# s3-21 정제 결과 저장
# TODO: 분석 표본을 data/processed/ 에 저장 (코드로만 생성, 수동 편집 금지)
# TODO: 정제 로그 표(log)와 추적표 출력 → logs/정제로그.md 표 4·5에 옮김

## ④ 분석 (25점 — 기준선 비교 필수)
- **무엇을**: EDA·기술통계 → 분석 표본 → 모델 → 기준선 → 민감도 → 가설 A·B 판정과 각 예측 절.
- **왜**: 가설을 사전 고정된 규칙대로 판정하고, 예측 모델이 기준선보다 얼마나 나은지(또는 못한지) 정직하게 보고하려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 시간 분리(학습 t=2024·2025 / 테스트 t=2026), 누수 점검 assert, 기준선 2개와 같은 표, 민감도 표, RANDOM_STATE=42 재실행 일치

In [ ]:
# s4-00 분석 공통 규칙 (주석만)
# - 무작위 분할 금지. 학습 t=2024·2025, 테스트 t=2026. RANDOM_STATE=42
# - 고위험 컷(HIGH_PCT 백분위)은 학습 데이터로만 계산해 테스트에 그대로 적용
# - 누수 금지: 당해 연도의 계·사유·비율은 설명 변수로 쓰지 않음
# - 설명 변수: 대계열, 설립유형, 소재지유형, 전년도 재적 규모(구간). 소계열은 쓰지 않음(B 대상군 정의 전용)
# - A 판정용 모델: 전년도 탈락률 제외 / 예측 성능용 모델: 전년도 탈락률 추가

### ④-1 EDA·기술통계

In [ ]:
# s4-01 EDA: 재적·계 분포
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-01)'를 붙인다.
# 내용: 절대=인원(명)과 비중=비율(%)을 함께 본다. 범위 안 기준
# TODO

In [ ]:
# s4-02 기술통계 (가설 A, 예측 아님)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-02)'를 붙인다.
# 내용: 설립유형 × 수도권 4칸의 고위험 비율, 시·도별 순위
# TODO

### ④-2 분석 표본과 y

In [ ]:
# s4-03 y 정의
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-03)'를 붙인다.
# 내용: 학습기간 유효 학과의 HIGH_PCT 백분위 컷 → 테스트에 그대로 적용. 컷 값과 학습/테스트 양성 비율 출력
# TODO

In [ ]:
# s4-04 설명 변수와 누수 점검
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-04)'를 붙인다.
# 내용: 변수 목록 출력, 당해 연도 변수 미포함 assert
# TODO

### ④-3 가설 A: 전국 구조

In [ ]:
# s4-05 A 판정용 로지스틱 회귀 (statsmodels)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-05)'를 붙인다.
# 내용: 사립·비수도권 + 대계열 + 전년도 재적 규모. 오즈비와 95% CI 표 → 판정 규칙(맞음/부분/기각) 적용
# TODO

### ④-4 A 안의 예측 절

In [ ]:
# s4-06 모델 2개 학습
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-06)'를 붙인다.
# 내용: 구조 모델 / 구조 + 전년도 탈락률 모델 (학습 t=2024·2025)
# TODO

In [ ]:
# s4-07 기준선 2개
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-07)'를 붙인다.
# 내용: ① 전원 '고위험 아님' ② 전년도 고위험 지속
# TODO

In [ ]:
# s4-08 테스트 평가표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-08)'를 붙인다.
# 내용: 같은 k 정밀도(주), 정확도·정밀도·재현율·AUC·혼동행렬을 기준선과 한 표로. 임계값 0.5 vs k 기준. 판정 임계값 없음
# TODO

### ④-5 가설 B: 한국외대

In [ ]:
# s4-09 B 대상군 정의
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-09)'를 붙인다.
# 내용: 한국외대 + 소계열 in B_SUBFIELDS. 연도별 학과 수·재적 합계
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 매년 23개 학과
# TODO

In [ ]:
# s4-10 B 판정
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-10)'를 붙인다.
# 내용: 2024~2026 합산 비율(탈락 합계 ÷ 재적 합계, %) 대상군 vs 교내 다른 학과, 학과코드 단위 부트스트랩 95% 구간(RANDOM_STATE 고정) → 판정
# TODO

In [ ]:
# s4-11 B 보조 분석
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-11)'를 붙인다.
# 내용: 대상군 안에서 전년도 재적 하위 25%(소규모)가 더 높은지 (재적 대리 지표)
# TODO

In [ ]:
# s4-12 B 참고 해석 (가설 아님)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-12)'를 붙인다.
# 내용: 전국(한국외대 제외) 같은 소계열과 비교만. 미리 고정한 결과 문구 중 하나를 선택
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 비교군 8~9개교, 31~43개 학과
# TODO

### ④-6 B 안의 예측 절 (보조 참고)

In [ ]:
# s4-13 한국외대 예측
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-13)'를 붙인다.
# 내용: A 모델로 2026 공시 한국외대 학과의 다음 해 고위험 점수 → 우선 지원 후보 상위 k개. t=2026 테스트의 한국외대 학과(약 130개)에서 기준선 ②와 정밀도 비교. 미연결 학과 건수 공개
# TODO

### ④-7 민감도 (사전 고정 ①~⑤)

In [ ]:
# s4-14 민감도 표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-14)'를 붙인다.
# 내용: ① B에 국제지역학 소계열 추가 ② 전체 표본으로 A 재추정 ③ 최소 재적 10·30·50 ④ 고위험 컷 상위 20%·30% ⑤ k 기준 vs 임계값 0.5(비용 가정 명시). 결론이 뒤집히는지 한 표로
# TODO

### ④-8 판정 요약

In [ ]:
# s4-15 가설 판정 표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-15)'를 붙인다.
# 내용: 가설 | 판정(맞음/부분/기각) | 근거 수치 | 셀 ID | 판정에 따른 행동
# TODO

## ⑤ 발표 (시각화·권고·근거·한계)
- **무엇을**: 필수 차트 3종을 그리고, 권고·근거·한계와 '믿기 어려운 이유'를 정리한다.
- **왜**: 차트마다 '그래서 무엇' 한 줄과 출처를 붙여 슬라이드(`slides/발표_구성.md`)로 바로 옮기려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 그림 파일(figures/), 축 단위(명/%), 그림 아래 출처 줄, 슬라이드 구성표의 ⚠ 0개

In [ ]:
# 차트 ① 탈락 인원(명) × 탈락률(%) 교차 산점도 — 소규모 학과 비율 과대 확인
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-01)'를 붙인다.
# TODO: 그림 저장 → figures/, 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [ ]:
# 차트 ② 한국외대 학과군별 탈락률 비교 막대 (가설 B, 부트스트랩 구간)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-02)'를 붙인다.
# TODO: 그림 저장 → figures/, 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [ ]:
# 차트 ③ 모델 vs 기준선 정밀도 비교 (같은 k, 예측 절)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-03)'를 붙인다.
# TODO: 그림 저장 → figures/, 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

### 권고·근거 (결과 확인 후 작성)
| 가설 | 판정 | 근거 수치 (셀 ID) | 권고 |
|---|---|---|---|
| A | | | |
| B | | | |

### 한계 (빠뜨리면 감점)
1. 테스트는 공시 2026 한 해뿐이다.
2. 학과 단위 집계라 개인 이탈 요인을 담지 못하며, 결과는 상관이지 인과가 아니다(A의 오즈비는 연관).
3. 폐지·통폐합·신설 학과가 연결에서 빠지는 생존 편향, 공시 정의·결측 패턴 변화. 한국외대 학과코드 변동(B는 군 단위 풀링).
- 추가: 표준분류는 학교별 신고에 의존해 같은 이름의 학과가 학교마다 다르게 분류될 수 있다. 우리는 학과를 재분류하지 않고 데이터가 준 분류를 그대로 쓴다. / "비인기"·"존폐 위기"는 판단 불가 / 캠퍼스 미구분 / 전국 비교군 소표본 / 학부·전공 이중 계상 가능성 `[확인 필요]`

### 믿기 어려운 이유 · 말할 수 없는 것 (결과 확인 후 작성)
- `[팀 입력]`

## 부록 A. 재현성 확인
- **무엇을**: 런타임 재시작 후 처음부터 모두 실행해 핵심 수치를 한 셀에 모아 출력한다.
- **왜**: 다른 계정에서 돌려도 같은 숫자가 나와야 한다(안내문 마지막 점검).
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: `logs/재현성_확인표.md`의 항목과 같은 이름으로 출력, 값이 다르면 '발견'으로 기록

In [ ]:
# sA-01 재현성 요약
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §부록 A 셀 sA-01)'를 붙인다.
# TODO: 원본 행×열, 범위 안 행 수, 분석 표본(학습/테스트), A 오즈비, B 차이·구간, 같은 k 정밀도, AUC, 기준선 ②, 실행 일시, 라이브러리 버전